# Introduction

In [1]:
from rdf_extract import GraphReader, PrefixStore, DataTree, Compiler

This is a demo for the package "rdf_extract". It contains the following classes:

- GraphReader: Loads a RDF graph into memory to interact with it. Can execute queries, rename nodes, extract subgraphs, and output triples as dataframes or dictionaries with native Python data types.
- PrefixStore: Stores prefixes and their urls as key-value pairs to support various operations based on prefixes (e.g. append prefixes to queries, drop prefixes from keys in dictionaries, etc.)
- DataTree: Class based on Json-Ld, which allows to express data as graphs or dictionaries interchangably. Subsets of the data can be selected, keys (or values) can be transformed. Supports various serializations (yaml, json, turtle).
- Compiler: Generic class that either takes a DataTree as input, returns a DataTree as output, or both. Schemas for expected input and output can be optionally provided for automatic validation of the transformation step.

## GraphReader

##### Load Data

In [ ]:
# Initialize a new GraphReader based on triples contained in a turtle file 
relative_filepath = "..//data//"

graph_reader = GraphReader(relative_filepath + "component catalog.ttl")

# Add triples from another graph to the GraphReader, make sure existing triples are not overwritten
graph_reader.load(relative_filepath + "pipeline.ttl")

##### Select and Display Data

In [ ]:
# Show all triples as a table:
graph_reader.get_triples()

# Show triples matching a pattern:
graph_reader.get_triples(pred = "rdf:type")
graph_reader.get_triples(sub = ":LdioPipeline")

# Check which prefixes are known by the graph reader:
graph_reader.prefix_store

# Execute a query
# A select query returns results as a dataframe
select_query = f"""
            SELECT ?step ?prev_step ?component
            WHERE {{
            ?step p-plan:isStepOfPlan :LdioPipeline .
            OPTIONAL {{?step p-plan:isPrecededBy ?prev_step .}}
            ?step tc:toBeCarriedOutByComponent ?component .
            }}
        """
graph_reader.execute_query(select_query)

##### Extract subgraphs from a bigger graph

In [ ]:
# A construct query returns results as a rdflib Graph. This graph can be turned into a new graph_reader
construct_query = f"""
            CONSTRUCT 
            {{ 
                ?step :follows ?prev_step . 
                ?step :uses ?component .
            }}
            WHERE 
            {{
            ?step p-plan:isStepOfPlan :LdioPipeline .
            OPTIONAL {{?step p-plan:isPrecededBy ?prev_step .}}
            ?step tc:toBeCarriedOutByComponent ?component .
            }}
        """

constructed_graph = graph_reader.execute_query(construct_query)
constructed_graph_reader = GraphReader(constructed_graph)
constructed_graph_reader.get_triples()

# You can also extract subgraphs from the GraphReader via graph traversal:
subgraph = graph_reader.extract_subgraph(
            ":LdioPipeline", direction="along", against="p-plan:isStepOfPlan", prune=["osw:hasUseLimitations", "osw:hasDependency"]
        )
# In this example, a subgraph is extracted from the node with uri ":LdioPipeline". 
# From there, it follows all edges originating from this node (direction="along") recursively. 
# "osw:hasUseLimitations" and "osw:hasDependency" will also be followed in the default direction, 
# however graph traversal is stopped at the destination of these edges and hence not recursively repeated (prune=["osw:hasUseLimitations", "osw:hasDependency"])
# The only exeception here are triples with the predicate p-plan:isStepOfPlan, where :LdioPipeline is the object of the triple.
# These will also be followed (against="p-plan:isStepOfPlan").
# You can see that "extract_subgraph" allows to extract subsets of data in bigger graph by defining which edges should be followed in 
# which direction, given a starting node. 

# Extract a subgraph as a dictionary via Json-Ld framing:
graph_reader.to_dict(":LdioPipeline")
# All edges originating from the provided uri are followed in the direction "along" and returned as Json-ld.

##### Other

In [ ]:
# Check whether a specific URI exists in the graph:
graph_reader.check_node_exists(":LdioPipeline")

# Rename a URI to different one in the graph:
graph_reader.rename(":LdioPipeline", ":NewName")

# You can also rename nodes via match patterns. This is ideal to target blind nodes and turn them to proper Uris
graph_reader.rename(":LdioPipeline tc:storedConfig ?target.", ":LdioPipelineConfig")

# Return the graph stored in the graph reader as rdflib graph
graph_reader.copy_graph()

## PrefixStore

In [ ]:
# To be continued...